# NYCU Big Data Assignment 2 — November 7 revision
**Ella Tso / 113700058. Deadline: November 21, 23:55 Taiwan. Forecast: November 22–29.**

Run with **Runtime → Run all**. No GPU is needed. All work files live in your Google Drive (`MyDrive/asg2_work`), so a disconnected session resumes: run all again and verified hours are skipped.

Pipeline: download GH Archive hours → unchanged official `preprocess.py` (8 tables) → fixed ablations → extra analyses and slide figures → checked CSV → independent rebuild/refit reproduction check → run bundle.

Executed on 2026-11-08 for `[2026-09-01, 2026-11-07)` UTC: all 1,608 hours downloaded, the reproduction check passed with byte-identical CSV, and a separate re-run on another machine produced a byte-identical model checkpoint.

Data Source: https://www.gharchive.org/ . Only the resulting eight tables are used. On failure, the last lines of `WORK/logs/latest_command.log` identify the cause.


In [ ]:
import os, subprocess, pathlib, sys
from google.colab import drive
drive.mount('/content/drive')
os.environ['OMP_NUM_THREADS']='1'
os.environ['OPENBLAS_NUM_THREADS']='1'
PROJECT=pathlib.Path('/content/NYCU-BigData-Assignment2')
BRANCH='audit-baselines-20261102'
if not PROJECT.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ellatso/NYCU-BigData-Assignment2.git',str(PROJECT)],check=True)
else:
    dirty=subprocess.check_output(['git','-C',str(PROJECT),'status','--porcelain','--untracked-files=no'],text=True).strip()
    if dirty:
        raise RuntimeError('Tracked project files were edited locally. Save your edits before updating; no files were overwritten.\n'+dirty)
    subprocess.run(['git','-C',str(PROJECT),'pull','--ff-only','origin',BRANCH],check=True)
os.chdir(PROJECT)
subprocess.run(['git','rev-parse','HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements_revised.txt'],check=True)
if subprocess.run(['which','dot'],capture_output=True).returncode:
    subprocess.run(['apt-get','-qq','install','-y','graphviz'],check=True)  # schema/framework figures


## Choose source hours
Dates below are UTC; end is exclusive. The default permits the documented October historical experiments. For a final refresh, extend END_UTC to the chosen completed observation cutoff before the November22 forecast. Keep all future test-week hours out of training. Every distinct data period receives a new prepared-data directory.

Colab local files are temporary. Preserve the downloaded run bundle and manifests before the session expires. For resumable downloads across sessions, set WORK to a folder in your own mounted Drive; the notebook does not mount or share Drive automatically.

In [ ]:
import hashlib, json, sys, shutil
from collections import deque
WORK=pathlib.Path('/content/drive/MyDrive/asg2_work')
START_UTC='2026-09-01T00:00:00+00:00'
END_UTC='2026-11-07T00:00:00+00:00'
STORAGE='cohort'  # exact official filter; use 'full' to retain original hour files
RAW=WORK/('raw_'+hashlib.sha256((START_UTC+STORAGE).encode()).hexdigest()[:10])
RUN_KEY=hashlib.sha256((START_UTC+END_UTC+STORAGE).encode()).hexdigest()[:12]
DATA=WORK/('prepared_'+RUN_KEY)
OUT=WORK/('run_'+RUN_KEY)
WORK.mkdir(parents=True,exist_ok=True)
def run(*args):
    logdir=WORK/'logs';logdir.mkdir(parents=True,exist_ok=True)
    logfile=logdir/'latest_command.log'
    tail=deque(maxlen=60)
    command=[sys.executable,'-u',*map(str,args)]
    with logfile.open('w') as log:
        log.write('Command: '+repr(command)+'\n');log.flush()
        with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
            for line in proc.stdout:
                print(line,end='',flush=True);log.write(line);log.flush();tail.append(line)
            code=proc.wait()
    if code:
        manifest=RAW/'download_manifest.json'
        if manifest.exists():
            rows=json.loads(manifest.read_text()).get('files',{})
            failures={k:v for k,v in rows.items() if v.get('status')!='ok'}
            print('First recorded non-successful source hours:')
            print(json.dumps(dict(list(failures.items())[:5]),indent=2))
        raise RuntimeError('Command failed (exit '+str(code)+'). Log: '+str(logfile)+'\n'+''.join(tail)) from None
print('Python:',sys.version,sys.executable)
print('Free disk GB:',round(shutil.disk_usage(WORK).free/1e9,2))
WORKERS=8
# Hours recorded as 'blocked' by an older downloader version (HTTP 403 / 1010 client signature) are retried.
mp=RAW/'download_manifest.json'
if mp.exists():
    m=json.loads(mp.read_text())
    stale=[k for k,v in m.get('files',{}).items() if v.get('status')=='blocked']
    for k in stale: del m['files'][k]
    if stale:
        mp.write_text(json.dumps(m,indent=2)); print('Retrying',len(stale),'previously blocked hours')
run('scripts/prepare_revised.py','download','--raw',RAW,'--start',START_UTC,'--end',END_UTC,'--storage',STORAGE,'--workers',WORKERS)


In [ ]:
from asg2.provenance import verify_data
if not (DATA/'provenance.json').exists():
    run('scripts/prepare_revised.py','build','--raw',RAW,'--out',DATA)
verify_data(DATA)
print('Verified new official preprocessing and all table hashes')


## Matched ablations for Q2 and Q3
The fixed protocol compares observed history, recent activity, multiple time windows, recency/trend removal, static relations, received interactions, collaborator removal and the availability clock. The primary dynamic relational prototype is fixed in advance; thresholds use calibration only and the best calibration reference is reported separately. F1 is explanatory evidence and has **zero direct grade weight** under the revision. Coverage limitations stay attached to every result.

In [ ]:
run('-m','asg2.revised','experiment','--data',DATA,'--out',OUT,'--config','config/revised.json')
run('scripts/summarize_revised.py','--data',DATA,'--out',OUT)
import pandas as pd
from IPython.display import display,Image
display(pd.read_csv(OUT/'slide_evidence.csv'))
display(pd.read_csv(OUT/'coverage.csv'))
display(Image(filename=str(OUT/'ablation.png')))


## Extra analyses and slide figures (Q1–Q4)
`scripts/analyze_revised.py` refits the same variants and first asserts that it reproduces `metrics.csv` exactly, then writes: weekly volume and label prevalence, user-level paired bootstrap CIs, per-week and no-push diagnostics, TA reference-week check, feature-family gain, relational feature coverage, cohort closure and visibility delay. `scripts/figures_revised.py` draws every slide figure from those files.

In [ ]:
ANALYSIS=OUT/'analysis'; FIGS=OUT/'figures'
run('scripts/analyze_revised.py','--data',DATA,'--run',OUT,'--out',ANALYSIS)
run('scripts/figures_revised.py','--run',OUT,'--analysis',ANALYSIS,'--data',DATA,'--out',FIGS)
display(pd.read_csv(ANALYSIS/'bootstrap_development.csv').round(4))
display(pd.read_csv(ANALYSIS/'reference_week.csv').round(4))
for name in ['q1_schema_graph','q3_weekly_volume','q2_q3_ablation','q4_framework']:
    display(Image(filename=str(FIGS/f'{name}.png'),width=1100))


## Generate and independently reproduce the CSV
The current anonymous roster is checked by the pinned official evaluator. Unknown or not-yet-observed test users use a documented calibration prior; their IDs are not dropped. The output is an early forecast when the source cutoff precedes the final deadline. Keep the same data, code, libraries, model configuration and cutoff for reproduction.

The verification rebuilds all eight tables and fits the candidate models again. It can take time; it is part of the reproduction check, not an additional model search.

In [ ]:
run('-m','asg2.revised','predict','--data',DATA,'--out',OUT,'--refit','--student-id','113700058')
run('scripts/reproduce_revised.py','--raw',RAW,'--data',DATA,'--out',OUT,'--student-id','113700058')
print((OUT/'reproduction_checks.json').read_text())
print((OUT/'prediction_protocol.json').read_text())


In [ ]:
import zipfile
BUNDLE=WORK/('Asg2_Revised_Run_'+RUN_KEY+'.zip')
with zipfile.ZipFile(BUNDLE,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file():z.write(p,'artifacts_revised/'+p.relative_to(OUT).as_posix())
    for p in DATA.rglob('*'):
        if p.is_file():z.write(p,'prepared_data/'+p.relative_to(DATA).as_posix())
print('Save this run bundle:',BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
files.download(str(OUT/'output_113700058.csv'))


## Before E3 submission
- CSV: `output_113700058.csv`, latest anonymous roster, official format check, reproducible from the recorded source files and cutoff.
- English PDF: `Asg2_113700058.pdf`, teacher template, at most16 pages. Replace every pending-results statement with the actual new-run evidence before calling it final.
- Personally recorded MP4: `Asg2_113700058.mp4`,16–20 minutes. AI-generated video is prohibited. Cover the same Q1–Q4 evidence in both the slides and the narration.
- Public source and README acknowledge GH Archive. Keep raw/private data, models and individual predictions out of the public repository.
- Verify the actual E3 receipt; generating these files does not submit them.